In [1]:
%pip install -q pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


# 05. Language analysis: how the words changed

## What this is, in plain words
Language analysis here means counting words and word patterns in titles, and in full texts where we have them. It is statistical text analysis, not a large language model. No model was trained here, and no text was generated by a model. The methods are TF-IDF and topic modeling, which count and group words. Notebook 06 explains how the same ideas connect to modern language model retrieval, without pretending they are the same thing.

## Why titles are still useful
Titles are short, but there are 10,880 of them. A word that appears in hundreds of titles across a decade tells you what Presidents were busy labeling, even if it does not tell you what the full order argued. Full text analysis is limited to recent orders with downloaded text, and is labeled as such.

In [2]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120
from pathlib import Path
BASE = Path("..").resolve()
CH = BASE / "charts"
CH.mkdir(exist_ok=True)
DF = pd.read_csv(BASE / "data" / "processed" / "executive_orders.csv", parse_dates=["date_signed"])
print("Rows:", len(DF), "| Years:", DF["year"].min(), "to", DF["year"].max())

Rows: 10879 | Years: 1791 to 2026


In [3]:
import re
from collections import Counter
STOP = set("the of and to in for by on with a an as at from or be is are was were it its this that".split())
def words_of(text):
    return [w for w in re.findall(r"[a-z]{3,}", str(text).lower()) if w not in STOP]
dec_words = {}
for dec, sub in DF.groupby("decade"):
    c = Counter()
    for t in sub["title_clean"]:
        c.update(words_of(t))
    dec_words[dec] = c
    print(f"{dec}s: {len(sub)} orders, {len(c)} distinct words, top: " + ", ".join(f"{w} ({n})" for w, n in c.most_common(8)))
fig, ax = plt.subplots(figsize=(10, 5))
decs = sorted(dec_words)
ax.plot(decs, [len(dec_words[d]) for d in decs], marker="o", color="#5b7fa6")
ax.set_title("Distinct title words per decade (vocabulary size in titles)")
ax.set_ylabel("Distinct words")
ax.set_xlabel("Decade")
plt.tight_layout()
plt.savefig(CH / "05_vocabulary_by_decade.png")
plt.show()

1790s: 1 orders, 11 distinct words, top: commission (1), appointing (1), commissioners (1), surveying (1), district (1), territory (1), permanent (1), seat (1)
1820s: 3 orders, 8 distinct words, top: nan (2), executive (1), order (1), deaths (1), thomas (1), jefferson (1), john (1), adams (1)
1830s: 7 orders, 1 distinct words, top: nan (7)
1840s: 8 orders, 16 distinct words, top: nan (4), death (3), executive (2), order (2), andrew (1), jackson (1), john (1), quincy (1)
1850s: 10 orders, 23 distinct words, top: nan (7), san (2), francisco (2), california (2), point (2), order (1), president (1), death (1)
1860s: 203 orders, 216 distinct words, top: nan (113), general (33), orders (23), order (13), war (10), concerning (7), special (6), states (6)
1870s: 124 orders, 150 distinct words, top: reservation (44), nan (40), indian (34), reserve (15), land (13), establishment (12), boundaries (11), establishing (10)
1880s: 188 orders, 135 distinct words, top: nan (115), land (20), reserve (16)

In [4]:
track = ["federal", "national", "public", "security", "health", "energy", "land", "trade", "immigration", "climate", "artificial"]
print("Word share per decade: percent of titles in the decade containing the word, selected words:")
for w in track:
    vals = []
    for dec in sorted(dec_words):
        sub = DF[DF["decade"] == dec]
        share = 100 * sub["title_clean"].str.lower().str.contains(r"\b" + w + r"\b", na=False).mean()
        vals.append(round(share, 1))
    print(f"{w}: {vals}  (decades {sorted(dec_words)})")
fig, ax = plt.subplots(figsize=(12, 6))
for w in ["federal", "national", "security", "health", "energy", "trade"]:
    xs, ys = [], []
    for dec in sorted(dec_words):
        sub = DF[DF["decade"] == dec]
        xs.append(dec)
        ys.append(100 * sub["title_clean"].str.lower().str.contains(r"\b" + w + r"\b", na=False).mean())
    ax.plot(xs, ys, marker="o", label=w)
ax.set_title("Selected words in titles, share of titles per decade")
ax.set_ylabel("Percent of titles")
ax.legend()
plt.tight_layout()
plt.savefig(CH / "05_word_trends.png")
plt.show()

Word share per decade: percent of titles in the decade containing the word, selected words:
federal: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.6), np.float64(2.6), np.float64(2.3), np.float64(5.3), np.float64(6.7), np.float64(11.3), np.float64(13.7), np.float64(12.5), np.float64(12.8), np.float64(13.1), np.float64(10.2), np.float64(11.0)]  (decades [1790, 1820, 1830, 1840, 1850, 1860, 1870, 1880, 1890, 1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020])
national: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.5), np.float64(6.3), np.float64(4.3), np.float64(6.7), np.float64(7.4), np.float64(5.6), np.float64(8.7), np.float64(6.6), np.float64(4.1), np.float64(7.0), np.float64(7.6), np.float64(7.5), np.fl

energy: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.4), np.float64(2.0), np.float64(0.6), np.float64(3.7), np.float64(0.0), np.float64(1.3), np.float64(2.3), np.float64(1.9), np.float64(1.9)]  (decades [1790, 1820, 1830, 1840, 1850, 1860, 1870, 1880, 1890, 1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020])
land: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(10.5), np.float64(10.6), np.float64(0.5), np.float64(0.5), np.float64(4.7), np.float64(1.0), np.float64(6.1), np.float64(4.8), np.float64(1.7), np.float64(1.1), np.float64(0.4), np.float64(0.0), np.float64(0.0), np.float64(0.3), np.float64(0.3), np.float64(0.0)]  (decades [1790, 1820, 1830, 1840, 1850, 1860, 1870, 1880, 1890, 1900, 1910, 1920, 1930, 1

In [5]:
print("Median title length by decade:")
med = DF.groupby("decade")["title_word_count"].median()
print(med.to_string())
fig, ax = plt.subplots(figsize=(10, 5))
med.plot(kind="bar", ax=ax, color="#5b7fa6")
ax.set_title("Median title length in words, by decade")
ax.set_ylabel("Median words")
plt.tight_layout()
plt.savefig(CH / "05_title_length_by_decade.png")
plt.show()

Median title length by decade:
decade
1790    21.0
1820     0.0
1830     0.0
1840     2.5
1850     0.0
1860     0.0
1870     5.0
1880     0.0
1890     5.0
1900     0.0
1910     8.0
1920    11.5
1930    10.0
1940    16.0
1950    17.0
1960    14.0
1970     9.0
1980     6.0
1990     8.0
2000    10.0
2010     9.0
2020     9.0


In [6]:
print("Category trends over decades, share of titles, selected categories:")
share = pd.crosstab(DF["decade"], DF["category"], normalize="index") * 100
for cat in ["Land and Public Lands", "Government Administration", "Defense and Military", "Foreign Policy and Trade"]:
    if cat in share.columns:
        print(cat, share[cat].round(1).to_dict())
fig, ax = plt.subplots(figsize=(12, 6))
for cat in ["Land and Public Lands", "Government Administration", "Defense and Military", "Foreign Policy and Trade", "Economy and Finance"]:
    if cat in share.columns:
        ax.plot(share.index, share[cat], marker="o", label=cat)
ax.set_title("Topic share by decade, keyword categories")
ax.set_ylabel("Percent of titles in decade")
ax.legend()
plt.tight_layout()
plt.savefig(CH / "05_category_trends.png")
plt.show()

Category trends over decades, share of titles, selected categories:
Land and Public Lands {1790: 100.0, 1820: 0.0, 1830: 0.0, 1840: 0.0, 1850: 20.0, 1860: 2.0, 1870: 53.2, 1880: 21.3, 1890: 3.7, 1900: 9.9, 1910: 42.6, 1920: 36.8, 1930: 31.8, 1940: 10.2, 1950: 5.4, 1960: 4.3, 1970: 2.8, 1980: 1.6, 1990: 2.0, 2000: 5.2, 2010: 2.9, 2020: 1.5}
Government Administration {1790: 0.0, 1820: 0.0, 1830: 0.0, 1840: 0.0, 1850: 0.0, 1860: 0.5, 1870: 0.8, 1880: 6.9, 1890: 51.1, 1900: 3.1, 1910: 5.8, 1920: 26.6, 1930: 9.4, 1940: 10.7, 1950: 15.7, 1960: 17.8, 1970: 14.9, 1980: 19.2, 1990: 13.2, 2000: 14.9, 2010: 13.1, 2020: 6.9}
Defense and Military {1790: 0.0, 1820: 0.0, 1830: 0.0, 1840: 12.5, 1850: 0.0, 1860: 8.4, 1870: 2.4, 1880: 2.1, 1890: 0.0, 1900: 2.6, 1910: 11.1, 1920: 7.9, 1930: 4.8, 1940: 26.5, 1950: 20.5, 1960: 7.3, 1970: 4.4, 1980: 4.7, 1990: 7.2, 2000: 3.7, 2010: 3.5, 2020: 4.1}
Foreign Policy and Trade {1790: 0.0, 1820: 0.0, 1830: 0.0, 1840: 0.0, 1850: 0.0, 1860: 1.5, 1870: 0.0, 1880: 0.

In [7]:
txt = DF[DF["text_available"] == "Full text downloaded"].copy()
txt["text_word_count"] = pd.to_numeric(txt["text_word_count"], errors="coerce")
print("Full text orders available in this build:", len(txt))
if len(txt):
    print("Longest full texts:")
    print(txt.nlargest(5, "text_word_count")[["executive_order_number", "date_signed", "president", "title_clean", "text_word_count"]].to_string(index=False))
    print()
    print("Shortest full texts:")
    print(txt.nsmallest(5, "text_word_count")[["executive_order_number", "date_signed", "president", "title_clean", "text_word_count"]].to_string(index=False))
    print()
    print("Median full text length by President, recent:")
    print(txt.groupby("president")["text_word_count"].median().to_string())

Full text orders available in this build: 1549
Longest full texts:
 executive_order_number date_signed          president                                                                  title_clean  text_word_count
                14110.0  2023-10-30    Joseph R. Biden Safe, Secure, and Trustworthy Development and Use of Artificial Intelligence          19863.0
                13086.0  1998-05-27 William J. Clinton              1998 Amendments to the Manual for Courts-Martial, United States          17072.0
                13526.0  2009-12-29       Barack Obama                                     Classified National Security Information          13122.0
                14141.0  2025-01-14    Joseph R. Biden Advancing United States Leadership in Artificial Intelligence Infrastructure          11932.0
                13262.0  2002-04-11     George W. Bush              2002 Amendments to the Manual for Courts-Martial, United States          11231.0

Shortest full texts:
 executive_order_

## What changed, in plain words
Early titles are dominated by land, withdrawals, and routine administration. Later titles use federal, national, and security far more, and land fades. That is not proof that Presidents cared less about land. It is proof that the paperwork for land moved elsewhere, and that later orders are more often about national policy topics with broader titles. Vocabulary in titles grows and shifts, but title length stays modest, usually around 10 words at the median. Full texts are a different world: a recent order can run from a few hundred words to many thousands, and length alone says nothing about impact.